# Economic Indicators + Control Flow in R
## Solution Notebook — Decision-Making on Macroeconomic Indicators

Complete solutions, vectorised alternates (`ifelse`), and a reusable policy-stance function.


## Flowchart

![Economic Indicators + Control Flow](economic_indicators_control_flow_flowchart.png)


## 0. Shared Data (same as skeleton)


In [ ]:
years <- 2017:2026
inflation <- c(2.1, 2.4, 1.8, 1.2, 4.7, 8.0, 4.1, 3.4, 2.9, 2.5)
gdp_growth <- c(2.3, 2.9, 2.3, -2.8, 5.9, 2.1, 2.5, 2.0, 1.8, 2.2)
unemployment <- c(4.4, 3.9, 3.7, 8.1, 5.4, 3.6, 3.7, 4.0, 4.2, 4.1)

macro_df <- data.frame(
  year         = 2022:2026,
  inflation    = c(8.0, 4.1, 3.4, 2.9, 2.5),
  gdp_growth   = c(1.9, 2.5, 2.1, 1.8, 2.0),
  unemployment = c(3.6, 3.7, 4.0, 4.2, 4.1),
  stringsAsFactors = FALSE
)
print(macro_df)


## 1. Simple `if` — Solutions


In [ ]:
# Task 1.1
latest <- tail(inflation, 1)
if (latest > 3.0) {
  print("Inflation still above comfort zone — monitor closely.")
}
# Output (with current data): the message is printed because 2.5 is NOT > 3? 
# Wait — latest is 2.5, so nothing prints. Change the threshold or the data to see the branch.
# Demonstration with a higher value:
if (8.0 > 3.0) {
  print("Inflation still above comfort zone — monitor closely.")
}


In [ ]:
# Task 1.2
if (macro_df$unemployment[1] < 4.0) {
  print("Labour market remains tight.")
}
# 3.6 < 4.0 → message is printed


## 2. `if` / `else` — Solutions


In [ ]:
# Task 2.1
avg_g <- mean(gdp_growth)
if (avg_g > 0) {
  print("Economy expanded on average over the decade.")
} else {
  print("Economy contracted on average over the decade.")
}
# mean is positive → first message


In [ ]:
# Task 2.2
policy_budget <- 100
n_priorities  <- 4
units_per_priority <- policy_budget / n_priorities

if (units_per_priority >= 25) {
  print("Adequate resources for each priority.")
} else {
  print("Resources stretched — prioritisation required.")
}
# 25 >= 25 → Adequate...


## 3. `else if` — Solutions


In [ ]:
# Task 3.1
latest_inf <- tail(inflation, 1)
if (latest_inf >= 5) {
  label <- "High inflation regime"
} else if (latest_inf >= 3 & latest_inf < 5) {
  label <- "Elevated inflation"
} else if (latest_inf >= 2 & latest_inf < 3) {
  label <- "Target-consistent"
} else {
  label <- "Low inflation / risk of undershoot"
}
print(label)
# 2.5 → Target-consistent


In [ ]:
# Task 3.2 — loop version
regime <- character(length(inflation))
for (i in seq_along(inflation)) {
  x <- inflation[i]
  if (x >= 5) {
    regime[i] <- "High inflation regime"
  } else if (x >= 3 & x < 5) {
    regime[i] <- "Elevated inflation"
  } else if (x >= 2 & x < 3) {
    regime[i] <- "Target-consistent"
  } else {
    regime[i] <- "Low inflation / risk of undershoot"
  }
}
print(data.frame(year = years, inflation = inflation, regime = regime))


In [ ]:
# Alternate — fully vectorised with nested ifelse (preferred in R)
regime_vec <- ifelse(inflation >= 5, "High inflation regime",
              ifelse(inflation >= 3, "Elevated inflation",
              ifelse(inflation >= 2, "Target-consistent",
                     "Low inflation / risk of undershoot")))
print(data.frame(year = years, inflation = inflation, regime = regime_vec))
# identical(regime, regime_vec) should be TRUE


## 4. Nested Conditionals — Solutions


In [ ]:
# Task 4.1
g <- tail(gdp_growth, 1)
u <- tail(unemployment, 1)

if (g > 0) {
  if (u < 4.0) {
    print("Goldilocks: positive growth + tight labour market.")
  } else {
    print("Growth positive but labour market softening.")
  }
} else {
  print("Negative or zero growth — prioritise stabilisation.")
}
# current values: g=2.2 >0, u=4.1 → second message


In [ ]:
# Task 4.2 — nested access logic
access_briefing      <- FALSE
access_detailed_model <- FALSE
experience_years     <- 6
current_hour         <- 14
possible_codeword    <- "phillips"
real_codeword        <- "phillips"

if (experience_years >= 5) {
  access_briefing <- TRUE
  print("Briefing access granted.")
  if (possible_codeword == real_codeword) {
    access_detailed_model <- TRUE
    print("Detailed model access granted.")
  }
} else if (experience_years >= 2 & current_hour < 17) {
  access_briefing <- TRUE
  print("Limited briefing access until 17:00.")
} else {
  print("Access denied — gain more experience or wait for office hours.")
}

print(paste("access_briefing:", access_briefing))
print(paste("access_detailed_model:", access_detailed_model))


## 5. More Practice — Solutions


In [ ]:
# Practice A — already shown above as regime_vec
regime_vec <- ifelse(inflation >= 5, "High inflation regime",
              ifelse(inflation >= 3, "Elevated inflation",
              ifelse(inflation >= 2, "Target-consistent",
                     "Low inflation / risk of undershoot")))
print(data.frame(year = years, inflation = inflation, regime = regime_vec))


In [ ]:
# Practice B — logical indexing (preferred)
high_inf_years <- macro_df[macro_df$inflation >= 3.0, ]
print(high_inf_years)

# Alternate (loop style — less idiomatic)
rows <- list()
for (i in 1:nrow(macro_df)) {
  if (macro_df$inflation[i] >= 3.0) {
    rows[[length(rows) + 1]] <- macro_df[i, ]
  }
}
high_inf_loop <- do.call(rbind, rows)
print(high_inf_loop)


## 6. Simulation — Full Solution + Function


In [ ]:
# Editable parameters
current_inflation <- 3.8
current_growth    <- 1.2
current_unemp     <- 4.5
inflation_target  <- 2.0
growth_floor      <- 1.0

# Decision tree
if (current_inflation > inflation_target + 2 & current_growth > growth_floor) {
  stance <- "Aggressive tightening"
  why <- paste0("Inflation ", current_inflation, " is >2 pp above target and growth is solid.")
} else if (current_inflation > inflation_target) {
  stance <- "Moderate tightening"
  why <- paste0("Inflation ", current_inflation, " is above target but growth (", current_growth, ") is soft.")
} else if (abs(current_inflation - inflation_target) <= 0.5) {
  stance <- "Patient / data-dependent"
  why <- paste0("Inflation ", current_inflation, " is close to the ", inflation_target, " target.")
} else {
  stance <- "Easing bias"
  why <- paste0("Inflation ", current_inflation, " is below target — room for support.")
}

cat("Recommended stance:", stance, "\n")
cat("Justification:", why, "\n")


In [ ]:
# Reusable function version
recommend_stance <- function(inf, growth, target = 2.0, floor = 1.0) {
  if (inf > target + 2 && growth > floor) {
    list(stance = "Aggressive tightening",
         why = sprintf("Inflation %.1f is >2 pp above target and growth is solid.", inf))
  } else if (inf > target) {
    list(stance = "Moderate tightening",
         why = sprintf("Inflation %.1f is above target but growth (%.1f) is soft.", inf, growth))
  } else if (abs(inf - target) <= 0.5) {
    list(stance = "Patient / data-dependent",
         why = sprintf("Inflation %.1f is close to the %.1f target.", inf, target))
  } else {
    list(stance = "Easing bias",
         why = sprintf("Inflation %.1f is below target — room for support.", inf))
  }
}

# Test a few scenarios
print(recommend_stance(5.5, 2.0))
print(recommend_stance(3.2, 0.5))
print(recommend_stance(2.1, 1.5))
print(recommend_stance(1.4, 0.8))


## Key Takeaways

1. **Simple `if`** is perfect for threshold alerts on a single economic indicator.
2. **`if / else`** gives clean binary policy choices (expand vs contract, adequate vs stretched).
3. **`else if` chains** map naturally onto multi-tier regimes (high / elevated / target / low inflation).
4. **Nested conditionals** let you combine indicators (growth **and** unemployment) or implement multi-stage access rules.
5. Prefer **vectorised `ifelse`** (or `dplyr::case_when`) when labelling every row of a series; keep explicit `if` for single decisions and complex nested logic.
6. Wrap the final decision tree in a **function** so policy scenarios can be re-run with different numbers without touching the logic.
